# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/home/pedleyhuang/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

2026-05-20 18:49:32.046296: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-20 18:49:32.376134: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-20 18:49:32.376208: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-20 18:49:32.378026: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-20 18:49:32.552309: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-20 18:49:32.555403: I tensorflow/core/platform/cpu_feature_guard.cc:182] This Tens

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [9]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

X = df[feature_names].values
y = df["Class"].values
'''
np.array(X).shape
'''

(178, 13)

In [14]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.30,
    random_state = 42,
    stratify = y # keeps class proportionas equal in both splits
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

X_train: (124, 13), X_test: (54, 13)
y_train: (124,), y_test: (54,)


In [16]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
x_test = scaler.transform(X_test)

In [18]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_ohe = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_test_ohe  = tf.keras.utils.to_categorical(y_test,  num_classes=num_classes)

print(f"\ny_train_ohe shape : {y_train_ohe.shape}")
print(f"y_test_ohe  shape : {y_test_ohe.shape}")
print("\nFirst 5 one-hot labels (train):\n", y_train_ohe[:5])


y_train_ohe shape : (124, 3)
y_test_ohe  shape : (54, 3)

First 5 one-hot labels (train):
 [[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]]


In [19]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu',
                          input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax'),
])
model.summary()


2026-05-20 22:49:03.509347: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-20 22:49:03.513267: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2211] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [20]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train_ohe,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
13/13 [==============================] - 1s 16ms/step - loss: 0.8650 - accuracy: 0.7071 - val_loss: 0.7801 - val_accuracy: 0.6400
Epoch 2/20
13/13 [==============================] - 0s 5ms/step - loss: 0.6077 - accuracy: 0.9091 - val_loss: 0.5409 - val_accuracy: 0.8800
Epoch 3/20
13/13 [==============================] - 0s 4ms/step - loss: 0.4107 - accuracy: 0.9697 - val_loss: 0.3603 - val_accuracy: 0.9200
Epoch 4/20
13/13 [==============================] - 0s 5ms/step - loss: 0.2729 - accuracy: 0.9899 - val_loss: 0.2462 - val_accuracy: 0.9200
Epoch 5/20
13/13 [==============================] - 0s 4ms/step - loss: 0.1872 - accuracy: 0.9899 - val_loss: 0.1801 - val_accuracy: 0.9200
Epoch 6/20
13/13 [==============================] - 0s 5ms/step - loss: 0.1334 - accuracy: 0.9899 - val_loss: 0.1428 - val_accuracy: 0.9200
Epoch 7/20
13/13 [==============================] - 0s 4ms/step - loss: 0.1002 - accuracy: 0.9899 - val_loss: 0.1259 - val_accuracy: 0.9600
Epoch 8/20
13/13 [=

In [21]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

loss, acc = model.evaluate(X_test, y_test_ohe, verbose=0)
print(f"\nTest accuracy : {acc:.4f}")

y_pred_probs = model.predict(X_test)
# predicted class indices
y_pred = np.argmax(y_pred_probs, axis=1)
# true class indices
y_true = np.argmax(y_test_ohe,   axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred,
      target_names=["Class 0", "Class 1", "Class 2"]))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


Test accuracy : 0.3333
2/2 [==============================] - 0s 3ms/step

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.33      1.00      0.50        18
     Class 1       0.00      0.00      0.00        21
     Class 2       0.00      0.00      0.00        15

    accuracy                           0.33        54
   macro avg       0.11      0.33      0.17        54
weighted avg       0.11      0.33      0.17        54

Confusion Matrix:
[[18  0  0]
 [21  0  0]
 [15  0  0]]


/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,

In [23]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes
import os
converter    = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

path = "model_base.tflite"
with open(path, "wb") as f:
    f.write(tflite_model)

size_kb = os.path.getsize(path) / 1024
print(f"\nTFLite model saved → {path}  ({size_kb:.2f} KB)")


INFO:tensorflow:Assets written to: /tmp/tmpr5ag0ko1/assets


INFO:tensorflow:Assets written to: /tmp/tmpr5ag0ko1/assets



TFLite model saved → model_base.tflite  (14.07 KB)


2026-05-20 22:52:37.778100: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:52:37.778168: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:52:37.778376: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpr5ag0ko1
2026-05-20 22:52:37.779259: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:52:37.779275: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpr5ag0ko1
2026-05-20 22:52:37.781711: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:52:37.818436: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpr5ag0ko1
2026-05-20 22:52:37.828834: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 50459 m

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [26]:
# helper method
def file_size_kb(path):
    return os.path.getsize(path) / 1024

def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type  = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

        interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_is_quantized  = input_details['dtype']  in (np.int8, np.uint8)
    output_is_quantized = output_details['dtype'] in (np.int8, np.uint8)

    y_true = np.argmax(y_test_cat, axis=1)
    y_pred = []

    for i in range(len(X_test)):
        sample = X_test[i:i + 1].astype(np.float32)   # shape (1, 13)

        # Quantize input if the model expects integers
        if input_is_quantized:
            scale, zero_point = input_details['quantization']
            sample = np.round(sample / scale + zero_point).astype(input_details['dtype'])

        interpreter.set_tensor(input_details['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])   # shape (1, 3)

        # Dequantize output if the model returns integers
        if output_is_quantized:
            scale, zero_point = output_details['quantization']
            output = (output.astype(np.float32) - zero_point) * scale

        y_pred.append(np.argmax(output))

    y_pred = np.array(y_pred)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    print(f"\n{'='*55}")
    print(f"  {quant_type.upper()} TFLite model → {filename}")
    print(f"  File size : {file_size_kb(filename):.2f} KB")
    print(f"{'='*55}")
    print(classification_report(y_true, y_pred,
          target_names=["Class 0", "Class 1", "Class 2"]))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


In [27]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quantize_and_evaluate(model, X_test, y_test_ohe, 'int8',    'model_int8.tflite')


INFO:tensorflow:Assets written to: /tmp/tmp78yi_yiw/assets


INFO:tensorflow:Assets written to: /tmp/tmp78yi_yiw/assets
/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 23:04:10.016421: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:04:10.016537: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.



INT8 TFLite model size: 5.74 KB

  INT8 TFLite model → model_int8.tflite
  File size : 5.74 KB
              precision    recall  f1-score   support

     Class 0       0.28      0.50      0.36        18
     Class 1       0.32      0.33      0.33        21
     Class 2       0.00      0.00      0.00        15

    accuracy                           0.30        54
   macro avg       0.20      0.28      0.23        54
weighted avg       0.22      0.30      0.25        54

Confusion Matrix:
[[ 9  9  0]
 [14  7  0]
 [ 9  6  0]]


2026-05-20 23:04:10.016856: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp78yi_yiw
2026-05-20 23:04:10.017854: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:04:10.017895: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp78yi_yiw
2026-05-20 23:04:10.020672: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:04:10.057402: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp78yi_yiw
2026-05-20 23:04:10.066638: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 49781 microseconds.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is

In [28]:
quantize_and_evaluate(model, X_test, y_test_ohe, 'float16', 'model_float16.tflite')


INFO:tensorflow:Assets written to: /tmp/tmp57hk0fpn/assets


INFO:tensorflow:Assets written to: /tmp/tmp57hk0fpn/assets



FLOAT16 TFLite model size: 8.95 KB

  FLOAT16 TFLite model → model_float16.tflite
  File size : 8.95 KB
              precision    recall  f1-score   support

     Class 0       0.33      1.00      0.50        18
     Class 1       0.00      0.00      0.00        21
     Class 2       0.00      0.00      0.00        15

    accuracy                           0.33        54
   macro avg       0.11      0.33      0.17        54
weighted avg       0.11      0.33      0.17        54

Confusion Matrix:
[[18  0  0]
 [21  0  0]
 [15  0  0]]


2026-05-20 23:04:14.895966: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:04:14.896028: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 23:04:14.896233: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp57hk0fpn
2026-05-20 23:04:14.897166: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:04:14.897182: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp57hk0fpn
2026-05-20 23:04:14.900073: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:04:14.935367: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp57hk0fpn
2026-05-20 23:04:14.945171: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 48937 m

In [29]:
quantize_and_evaluate(model, X_test, y_test_ohe, 'dynamic', 'model_dynamic.tflite')

INFO:tensorflow:Assets written to: /tmp/tmptw7cppuz/assets


INFO:tensorflow:Assets written to: /tmp/tmptw7cppuz/assets
2026-05-20 23:04:17.671766: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:04:17.671845: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.



DYNAMIC TFLite model size: 8.17 KB

  DYNAMIC TFLite model → model_dynamic.tflite
  File size : 8.17 KB
              precision    recall  f1-score   support

     Class 0       0.33      1.00      0.50        18
     Class 1       0.00      0.00      0.00        21
     Class 2       0.00      0.00      0.00        15

    accuracy                           0.33        54
   macro avg       0.11      0.33      0.17        54
weighted avg       0.11      0.33      0.17        54

Confusion Matrix:
[[18  0  0]
 [21  0  0]
 [15  0  0]]


2026-05-20 23:04:17.672156: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmptw7cppuz
2026-05-20 23:04:17.672755: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:04:17.672793: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmptw7cppuz
2026-05-20 23:04:17.674655: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:04:17.708495: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmptw7cppuz
2026-05-20 23:04:17.718706: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 46551 microseconds.
/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` par

## Problem 1 - Part (c)

### Pruning

In [30]:
# substitution in step 2 for readability
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

In [31]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

batch_size  = 8
epochs      = 10
dataset_size = X_train.shape[0]                          # number of training samples

# Total gradient steps the pruning schedule spans
end_step = np.ceil(dataset_size / batch_size).astype(int) * epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity = 0.50,   # start with 50 % of weights zeroed
    final_sparsity   = 0.70,   # ramp up to 70 % zeroed by end_step
    begin_step       = 0,
    end_step         = end_step
)

In [32]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

pruned_model = tf.keras.Sequential([
    prune_low_magnitude(
        tf.keras.layers.Dense(64, activation='relu',
                              input_shape=(X_train.shape[1],)),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        tf.keras.layers.Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        tf.keras.layers.Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule
    ),
])
pruned_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 3 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 4 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 5 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [33]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep(),   # updates masks after every batch
]

history_pruned = pruned_model.fit(
    X_train, y_train_ohe,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 2s 15ms/step - loss: 0.9340 - accuracy: 0.5556 - val_loss: 0.7780 - val_accuracy: 0.7600
Epoch 2/10
13/13 [==============================] - 0s 5ms/step - loss: 0.6654 - accuracy: 0.8182 - val_loss: 0.5658 - val_accuracy: 0.8400
Epoch 3/10
13/13 [==============================] - 0s 6ms/step - loss: 0.4637 - accuracy: 0.9596 - val_loss: 0.3935 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 6ms/step - loss: 0.3104 - accuracy: 0.9899 - val_loss: 0.2764 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 5ms/step - loss: 0.2119 - accuracy: 0.9899 - val_loss: 0.2049 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 5ms/step - loss: 0.1472 - accuracy: 0.9899 - val_loss: 0.1727 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 5ms/step - loss: 0.1086 - accuracy: 0.9899 - val_loss: 0.1537 - val_accuracy: 0.9200
Epoch 8/10
13/13 [=

In [34]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter    = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # also quantize for max compression
tflite_pruned = converter.convert()

pruned_path = "model_pruned.tflite"
with open(pruned_path, "wb") as f:
    f.write(tflite_pruned)

print(f"\nPruned TFLite model saved → {pruned_path}")
print(f"File size: {file_size_kb(pruned_path):.2f} KB")


INFO:tensorflow:Assets written to: /tmp/tmpkfxm3q59/assets


INFO:tensorflow:Assets written to: /tmp/tmpkfxm3q59/assets
2026-05-20 23:16:29.544473: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:16:29.544548: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_depend


Pruned TFLite model saved → model_pruned.tflite
File size: 8.24 KB


ency.
2026-05-20 23:16:29.544952: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpkfxm3q59
2026-05-20 23:16:29.545488: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:16:29.545500: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpkfxm3q59
2026-05-20 23:16:29.546917: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:16:29.565701: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpkfxm3q59
2026-05-20 23:16:29.571470: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 26517 microseconds.


In [35]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

y_true          = np.argmax(y_test_ohe, axis=1)
y_pred_probs    = stripped_model.predict(X_test)
y_pred          = np.argmax(y_pred_probs, axis=1)

print(f"\n{'='*55}")
print(f"  PRUNED model evaluation (sparsity 50 % → 70 %)")
print(f"{'='*55}")
print(classification_report(y_true, y_pred,
      target_names=["Class 0", "Class 1", "Class 2"]))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 3ms/step

  PRUNED model evaluation (sparsity 50 % → 70 %)
              precision    recall  f1-score   support

     Class 0       0.33      1.00      0.50        18
     Class 1       0.00      0.00      0.00        21
     Class 2       0.00      0.00      0.00        15

    accuracy                           0.33        54
   macro avg       0.11      0.33      0.17        54
weighted avg       0.11      0.33      0.17        54

Confusion Matrix:
[[18  0  0]
 [21  0  0]
 [15  0  0]]


/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,

## Problem 1 - Part (d)

### Knowledge Distillation

In [36]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu',
                          input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax'),
], name='student')

student_model.summary()

Model: "student"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 32)                448       
                                                                 
 dense_7 (Dense)             (None, 16)                528       
                                                                 
 dense_8 (Dense)             (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [37]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_preds_soft = model.predict(X_train)          # shape (N, 3)  — 'model' = trained teacher

4/4 [==============================] - 0s 2ms/step


In [40]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

y_train_combined = np.concatenate([y_train_ohe, teacher_preds_soft], axis=1)

ALPHA = 0.5   # weight between hard and soft loss; 0 = pure distillation, 1 = pure CE

def distillation_loss(y_true_combined, y_pred):
    # Split the 6-column tensor back into its two halves
    y_true_hard = y_true_combined[:, :3]   # original one-hot ground truth
    y_true_soft = y_true_combined[:, 3:]   # teacher's soft probability output

    # Hard loss: standard cross-entropy against the true labels
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)

    # Soft loss: student learns to mimic the teacher's output distribution
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    # Weighted combination — alpha balances how much we trust the teacher
    return ALPHA * hard_loss + (1 - ALPHA) * soft_loss

In [41]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

history_kd = student_model.fit(
    X_train, y_train_combined,   # feed the 6-column combined label
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 1s 18ms/step - loss: 1.2309 - accuracy: 0.2020 - val_loss: 1.1589 - val_accuracy: 0.2000
Epoch 2/10
13/13 [==============================] - 0s 6ms/step - loss: 1.0741 - accuracy: 0.3636 - val_loss: 1.0457 - val_accuracy: 0.3200
Epoch 3/10
13/13 [==============================] - 0s 6ms/step - loss: 0.9593 - accuracy: 0.4646 - val_loss: 0.9556 - val_accuracy: 0.4000
Epoch 4/10
13/13 [==============================] - 0s 6ms/step - loss: 0.8649 - accuracy: 0.5657 - val_loss: 0.8676 - val_accuracy: 0.6800
Epoch 5/10
13/13 [==============================] - 0s 6ms/step - loss: 0.7811 - accuracy: 0.6970 - val_loss: 0.7766 - val_accuracy: 0.7200
Epoch 6/10
13/13 [==============================] - 0s 6ms/step - loss: 0.6932 - accuracy: 0.8081 - val_loss: 0.6866 - val_accuracy: 0.8400
Epoch 7/10
13/13 [==============================] - 0s 6ms/step - loss: 0.6112 - accuracy: 0.8788 - val_loss: 0.5901 - val_accuracy: 0.9200
Epoch 8/10
13/13 [=

In [42]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_kd = converter.convert()

kd_path = "model_kd.tflite"
with open(kd_path, "wb") as f:
    f.write(tflite_kd)

print(f"\nKnowledge Distillation TFLite model saved → {kd_path}")
print(f"File size: {file_size_kb(kd_path):.2f} KB")


INFO:tensorflow:Assets written to: /tmp/tmpjmphax9z/assets


INFO:tensorflow:Assets written to: /tmp/tmpjmphax9z/assets
2026-05-20 23:29:46.449709: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:29:46.449802: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.



Knowledge Distillation TFLite model saved → model_kd.tflite
File size: 6.04 KB


2026-05-20 23:29:46.450073: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpjmphax9z
2026-05-20 23:29:46.451097: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:29:46.451115: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpjmphax9z
2026-05-20 23:29:46.453927: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:29:46.485795: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpjmphax9z
2026-05-20 23:29:46.495684: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 45610 microseconds.


In [43]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

y_true           = np.argmax(y_test_ohe, axis=1)
y_pred_probs_kd  = student_model.predict(X_test)
y_pred_kd        = np.argmax(y_pred_probs_kd, axis=1)

print(f"\n{'='*55}")
print(f"  KNOWLEDGE DISTILLATION student model evaluation")
print(f"{'='*55}")
print(classification_report(y_true, y_pred_kd,
      target_names=["Class 0", "Class 1", "Class 2"]))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred_kd))

2/2 [==============================] - 0s 3ms/step

  KNOWLEDGE DISTILLATION student model evaluation
              precision    recall  f1-score   support

     Class 0       0.33      1.00      0.50        18
     Class 1       0.00      0.00      0.00        21
     Class 2       0.00      0.00      0.00        15

    accuracy                           0.33        54
   macro avg       0.11      0.33      0.17        54
weighted avg       0.11      0.33      0.17        54

Confusion Matrix:
[[18  0  0]
 [21  0  0]
 [15  0  0]]


/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/pedleyhuang/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,

## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction (Covered as a discussion in the companion PDF)

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [ ]:
# <-- (if needed) Enter your code here <--#

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
